# Cell 1: Install Required Libraries
!pip install -U transformers accelerate bitsandbytes

In [ ]:
# Cell 1: Install Required Libraries
!pip install -U transformers accelerate bitsandbytes

In [ ]:
# Cell 3: Main Generation Script
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# ---------------------------------------------------------------------------------

In [ ]:
# Cell 3: Main Generation Script
import torch
import gc
import random
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


Output Generation

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import torch.nn.functional as F
from tqdm.auto import tqdm

In [ ]:
STUDENT_BASE_PATH = "google/gemma-3-270m-it"
TEACHER_MODEL_PATH = "google/gemma-3-12b-it"

In [ ]:
print("\n3. Loading Teacher Model (12B)...")
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_PATH)
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_PATH,
    device_map="auto",
    # Pass 'load_in_4bit=True' here if you run out of memory!
    torch_dtype=torch.bfloat16
)
teacher_model.eval()


3. Loading Teacher Model (12B)...


config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi

In [ ]:
print("\n2. Loading Base Student Model...")
base_student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_BASE_PATH)
base_student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_BASE_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
base_student_model.eval()


2. Loading Base Student Model...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [ ]:
print("Loading new model...")
new_model_path = "/content/Distilled_Model"
new_tokenizer = AutoTokenizer.from_pretrained(new_model_path)
new_model = AutoModelForCausalLM.from_pretrained(
    new_model_path,
    device_map="auto",
    dtype=torch.bfloat16
)
new_model.eval()
print("New model loaded successfully.")

Loading new model...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

New model loaded successfully.


In [ ]:
# --- Re-load models and tokenizers within this cell for robustness ---
STUDENT_BASE_PATH = "google/gemma-3-270m-it"
TEACHER_MODEL_PATH = "google/gemma-3-12b-it"
new_model_path = "/content/Distilled_Model"

print("Re-loading Teacher Model (12B)...")
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_PATH)
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
teacher_model.eval()

print("Re-loading Base Student Model...")
base_student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_BASE_PATH)
base_student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_BASE_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
base_student_model.eval()

print("Re-loading New Model...")
new_tokenizer = AutoTokenizer.from_pretrained(new_model_path)
new_model = AutoModelForCausalLM.from_pretrained(
    new_model_path,
    device_map="auto",
    dtype=torch.bfloat16
)
new_model.eval()
print("All models re-loaded successfully.")
# --------------------------------------------------------------------

Re-loading Teacher Model (12B)...


config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Re-loading New Model...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

All models re-loaded successfully.


In [ ]:
  import pandas as pd
import torch
from tqdm.auto import tqdm
from pathlib import Path
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

# Enable tqdm for pandas apply functions
tqdm.pandas()

# Define paths
INPUT_CSV_PATH = Path("/content/Test_Data_15/test_ner_prompts_sampled_15.csv")
OUTPUT_CSV_PATH = Path("/content/Test_Data_15/test_ner_prompts_sampled_15_output_all_models.csv")

# Ensure output directory exists
OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

# Configuration for generation
MAX_NEW_TOKENS = 512

# --- Re-load models and tokenizers within this cell for robustness ---
STUDENT_BASE_PATH = "google/gemma-3-270m-it"
TEACHER_MODEL_PATH = "google/gemma-3-12b-it"
new_model_path = "/content/Distilled_Model"

print("Re-loading Teacher Model (12B)...")
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_PATH)
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
teacher_model.eval()

print("Re-loading Base Student Model...")
base_student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_BASE_PATH)
base_student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_BASE_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
base_student_model.eval()

print("Re-loading New Model...")
new_tokenizer = AutoTokenizer.from_pretrained(new_model_path)
new_model = AutoModelForCausalLM.from_pretrained(
    new_model_path,
    device_map="auto",
    dtype=torch.bfloat16
)
new_model.eval()
print("All models re-loaded successfully.")
# --------------------------------------------------------------------

# Regex to find JSON blocks in markdown
_JSON_BLOCK_RE = re.compile(r"```(?:json)?\s*([\s\S]*?)```", re.IGNORECASE)

def extract_json_from_output(raw_text: str) -> str:
    """
    Extracts JSON string from a text, handling markdown code blocks.
    If no JSON block is found, it attempts to parse the whole text as JSON.
    Returns a JSON string (or the raw extracted text if not valid JSON).
    """
    match = _JSON_BLOCK_RE.search(raw_text)
    if match:
        json_str = match.group(1).strip()
    else:
        json_str = raw_text.strip()

    # Attempt to validate if it's actual JSON. If not, return the extracted string.
    try:
        json.loads(json_str)
        return json_str
    except json.JSONDecodeError:
        return json_str # Return the extracted string even if it's not valid JSON

def generate_single_output(model, tokenizer, prompt):
    # Save original padding_side and restore it later
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, # Use greedy decoding for deterministic, clean JSON output
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id # Fallback if pad_token is None
        )

    # Restore original padding_side
    tokenizer.padding_side = original_padding_side

    input_length = inputs.input_ids.shape[1]
    gen_text = tokenizer.decode(output_ids[0][input_length:], skip_special_tokens=True).strip()
    return extract_json_from_output(gen_text)

print(f"Loading prompts from {INPUT_CSV_PATH}")
# Load all prompts first, then take the head to preserve original columns if needed.
all_prompts_df = pd.read_csv(INPUT_CSV_PATH)

print(f"Generating outputs for {len(all_prompts_df)} prompts, one by one...")

# Generate outputs for each model, one by one
print("Generating Teacher Model Outputs...")
all_prompts_df['Teacher Model Output'] = all_prompts_df['prompt'].progress_apply(lambda p: generate_single_output(teacher_model, teacher_tokenizer, p))

print("Generating Base Student Model Outputs...")
all_prompts_df['Base Student Model Output'] = all_prompts_df['prompt'].progress_apply(lambda p: generate_single_output(base_student_model, base_student_tokenizer, p))

print("Generating New Model Outputs...")
all_prompts_df['New Model Output'] = all_prompts_df['prompt'].progress_apply(lambda p: generate_single_output(new_model, new_tokenizer, p))

# Save the updated DataFrame to a new CSV file
all_prompts_df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"All outputs for {len(all_prompts_df)} records saved to {OUTPUT_CSV_PATH}")

Re-loading Teacher Model (12B)...


Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

Re-loading Base Student Model...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Re-loading New Model...


...

Generating Base Student Model Outputs...


  0%|          | 0/15 [00:00<?, ?it/s]

Generating New Model Outputs...


  0%|          | 0/15 [00:00<?, ?it/s]

All outputs for 15 records saved to /content/Test_Data_15/test_ner_prompts_sampled_15_output_all_models.csv


In [ ]:
import os
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import gc

# Define model paths
TEACHER_MODEL_ID = "google/gemma-3-12b-it"
BASE_STUDENT_MODEL_ID = "google/gemma-3-270m-it"
# Update this to point to the actual saved directory of your distilled model
DISTILLED_MODEL_DIR = "/content/Distill"

INPUT_CSV = r"/content/Test_Data/test_ner_prompts_sampled_100.csv"
OUTPUT_CSV = r"/content/Test_Data/test_ner_prompts_sampled_100_outputs.csv"

def generate_outputs_for_model(model_name_or_path, prompts, device):
    """Loads a model, generates outputs for all prompts, and unloads the model to free VRAM."""
    print(f"\n[{model_name_or_path}] Loading tokenizer and model...")
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name_or_path,
        torch_dtype=torch.bfloat16,
        device_map="auto" if "12b" in model_name_or_path.lower() else None
    )

    if "12b" not in model_name_or_path.lower():
        model = model.to(device)

    model.eval()

    outputs = []
    print(f"[{model_name_or_path}] Generating outputs...")

    for prompt in tqdm(prompts, desc=f"Generating ({model_name_or_path.split('/')[-1]})"):
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                use_cache=True,
            )

        gen_ids = out[0][inputs["input_ids"].shape[1]:]
        gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
        outputs.append(gen_text.strip())

    # Free VRAM
    print(f"[{model_name_or_path}] Unloading model and clearing VRAM...")
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return outputs

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Load the data
    print(f"Loading data from {INPUT_CSV}")
    df = pd.read_csv(INPUT_CSV)

    if "prompt" not in df.columns:
        raise ValueError("The input CSV must contain a 'prompt' column.")

    prompts = df["prompt"].tolist()

    # 2 & 3. Generate and store outputs sequentially to avoid OOM errors
    models_to_run = [
        ("Teacher_Output", TEACHER_MODEL_ID),
        ("Base_Student_Output", BASE_STUDENT_MODEL_ID),
        ("Distilled_Student_Output", DISTILLED_MODEL_DIR)
    ]

    for col_name, model_path in models_to_run:
        if "Distilled" in col_name and not os.path.exists(model_path):
            print(f"\n[WARNING] Distilled model path '{model_path}' not found! Skipping generation for distilled model.")
            df[col_name] = "MODEL_NOT_FOUND"
            continue

        df[col_name] = generate_outputs_for_model(model_path, prompts, device)

    # 4. Save the results
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSuccessfully generated outputs and saved to:\n{OUTPUT_CSV}")


main()

Loading data from /content/Test_Data/test_ner_prompts_sampled_100.csv

[google/gemma-3-12b-it] Loading tokenizer and model...


config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

[google/gemma-3-270m-it] Unloading model and clearing VRAM...

[/content/Distill] Loading tokenizer and model...


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

[/content/Distill] Generating outputs...


Generating (Distill): 100%|██████████| 100/100 [15:46<00:00,  9.47s/it]


[/content/Distill] Unloading model and clearing VRAM...

Successfully generated outputs and saved to:
/content/Test_Data/test_ner_prompts_sampled_100_outputs.csv
